# A carbon NMR shift: what the surroundings of one atom predict

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/igbaskin/cmf-qsar/blob/main/docs/tutorials/29_carbon_shifts.ipynb)

The chemical shift of a carbon atom in a 13C NMR spectrum belongs to that one atom. It is set
mostly by what is bonded to the carbon and by what sits one or two bonds further away. That makes
it the cleanest test of a local model: the sample is one atom and its surroundings, not a whole
molecule.

The previous notebook used such models to rank the atoms of a molecule. Here the target is a
number with units, the shift in ppm, and there is a method to compare with that spectroscopists
have used since 1978: the **HOSE code** (Bremser). It describes an atom sphere by sphere of bonds
and predicts the mean shift of the known atoms that share as many spheres with it as possible. It
needs no geometry and fits nothing. Any model that describes the surroundings of an atom has to
beat it before it says anything new.

The notebook asks three questions.

1. How is a HOSE lookup built, and how well does it predict on this set?
2. Does a model built on the fields around the carbon do better?
3. If it does, is that the **fields**, the charge, size, lipophilicity and hydrogen bonding of
   the atoms around the carbon? Or is it simply **where those atoms sit**? The test is to replace
   the five fields with five element channels (H, C, N, O and everything else) and keep
   everything else the same.

The data are 13C spectra from nmrshiftdb2 measured in deuterated chloroform, 400 molecules with
5,096 assigned carbons, kept in the repository. Running time: about three minutes.

In [1]:
# Run this cell first. On Google Colab it fetches the repository and installs the package;
# on a local checkout it only moves to the repository root, so that the example paths work.
import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.exists("cmf-qsar"):
        subprocess.run(["git", "clone", "-q", "https://github.com/igbaskin/cmf-qsar.git"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./cmf-qsar", "py3Dmol"], check=True)
    os.chdir("cmf-qsar")
else:
    while not os.path.exists("pyproject.toml") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")
print("working in", os.getcwd())

working in C:\Users\baski\work\claude\cmf


## The data, and the HOSE lookup

`AtomTargets` holds one row per measured carbon, with the molecule it belongs to. The folds hold
out **whole molecules**: two carbons of one molecule share most of their surroundings, and a
model tested on one of them after seeing the other would be tested on what it already knows.

The HOSE codes here are RDKit's Morgan environments. A carbon gets one identifier for itself
(0 bonds), one for itself with its neighbours (1 bond), and so on out to four bonds. To predict a
new carbon, the lookup takes the deepest sphere that some training carbon shares with it, and
returns the mean shift of the training carbons that share it.

The cell runs the lookup on every carbon, with ten folds, and prints for one carbon how many
training carbons share each sphere with it.

In [2]:
import json
import time

import numpy as np
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
sys.path.insert(0, "benchmarks")
import run_local as RL                         # the benchmark script of plan items LP2 and LP3

start = time.time()
mols, fields, targets, fieldset = RL.read_set("shifts")
folds = targets.folds(10, 0)
y = np.asarray(targets.y, float)
print("%d carbons in %d molecules; shifts from %.1f to %.1f ppm, standard deviation %.1f"
      % (len(targets), targets.n_measured_molecules, y.min(), y.max(), y.std()))

ids = RL.environment_ids(mols, targets, depth=4)
# the example carbon, chosen by a rule: of the carbons whose whole four-bond sphere occurs again
# somewhere in the set, the one whose two-bond sphere is the most common
_, deep_count = np.unique(ids[:, 4], return_inverse=True, return_counts=True)[1:]
_, inv2, cnt2 = np.unique(ids[:, 2], return_inverse=True, return_counts=True)
candidates = np.where(deep_count[np.unique(ids[:, 4], return_inverse=True)[1]] > 1)[0]
row = int(candidates[np.argmax(cnt2[inv2[candidates]])])
others = np.arange(len(y)) != row
print()
print("carbon %d of molecule %s, measured %.1f ppm" % (targets.atom[row],
                                                        fields[targets.molecule[row]].name, y[row]))
for k in range(5):
    same = others & (ids[:, k] == ids[row, k])
    print("  up to %d bonds: %4d other carbons share this sphere, mean shift %.1f ppm"
          % (k, same.sum(), y[same].mean() if same.any() else float("nan")))

pred, depth = RL.hose_lookup(ids, y, folds)
hose = RL.as_local_result("HOSE lookup, up to 4 bonds", pred, targets, folds)
print()
print("the lookup on every carbon, out of fold:")
print("  q2 %+.3f, error %.2f ppm, mean absolute error %.2f ppm" % (hose.q2, hose.error, hose.mae))
print("  the prediction was read at: " + ", ".join(
    "%d bonds %.0f %%" % (k, 100 * np.mean(depth == k)) for k in range(5) if np.any(depth == k)))
print("(%.0f s)" % (time.time() - start))

5096 carbons in 400 molecules; shifts from 7.0 to 220.5 ppm, standard deviation 53.4

carbon 13 of molecule 2276, measured 128.4 ppm
  up to 0 bonds: 1040 other carbons share this sphere, mean shift 124.8 ppm
  up to 1 bonds:  269 other carbons share this sphere, mean shift 128.0 ppm
  up to 2 bonds:  177 other carbons share this sphere, mean shift 128.2 ppm
  up to 3 bonds:   25 other carbons share this sphere, mean shift 128.8 ppm
  up to 4 bonds:    7 other carbons share this sphere, mean shift 128.8 ppm



the lookup on every carbon, out of fold:
  q2 +0.976, error 8.34 ppm, mean absolute error 4.93 ppm
  the prediction was read at: 0 bonds 8 %, 1 bonds 39 %, 2 bonds 29 %, 3 bonds 12 %, 4 bonds 12 %
(3 s)


**A lookup with no geometry and no fit reaches q2 +0.976**, an error of 8.34 ppm and a mean
absolute error of 4.93 ppm, on shifts that spread over 53 ppm.

The example carbon shows how it works. Its atom type alone is shared by 1,040 other carbons, whose
mean is 124.8 ppm. Each sphere further out narrows the group and moves the mean towards the
measured 128.4 ppm: 128.0 with the neighbours, 128.2 at two bonds, 128.8 at three and at four,
where 7 carbons are left.

With 400 molecules most new carbons find a match only one or two bonds out: 39 % of the
predictions are read at one bond and 29 % at two. Even so, the lookup is far ahead of the count
reference that plan item LP2 measured against (q2 +0.853, below). That reference is the right
kind of model, but not the reference a spectroscopist would choose.

## The surroundings of a ring carbon, computed here on part of the set

The strongest local model of plan item LP3 is built for carbons of a benzene ring. Each ring
carbon is put into one frame by a rule: the carbon at the origin, the bond that leaves the ring
along +x, the ring in the xy plane, both mirror images averaged. In that frame two environments
can be compared by the overlap kernel of the first notebook, field by field, and the model sees
**where** each atom around the carbon sits.

The cell fits that model twice on the same carbons and folds:

- with the five **fields** as its channels, as LP3 did;
- with five **element channels** in their place: one channel per element (H, C, N, O and all
  others together), each atom counted in its own element's channel with the same envelope weight.
  The positions are the same; only what the channels say about the atoms changes.

To keep the running time near two minutes, the cell uses the first 200 molecules of the file,
which hold about half of the ring carbons. The section after it reads the whole set from the
stored run.

In [3]:
from run_local_cofield import aromatic_samples
from cmfqsar.fields import FieldSet, build_fields
from cmfqsar.localalign import overlap_kernel, posed, ring_frames
from cmfqsar.localenv import choose_lam, environment, paired_interval

start = time.time()
rmols, rnames, rt = aromatic_samples(max_molecules=200)
rfolds = rt.folds(10, 0)
fs = FieldSet.create()
rfields = [build_fields(m, fs, name=n) for m, n in zip(rmols, rnames)]
print("%d ring carbons in %d molecules" % (len(rt), rt.n_measured_molecules))

rpred, _ = RL.hose_lookup(RL.environment_ids(rmols, rt, 4), rt.y, rfolds)
models = {"HOSE lookup": RL.as_local_result("HOSE lookup", rpred, rt, rfolds)}
for label in ("fields", "elements"):
    envs = []
    for r in range(len(rt)):
        env = environment(rfields[int(rt.molecule[r])], int(rt.atom[r]), RL.POSED_RHO)
        frames = ring_frames(env)                      # the frame comes from the geometry
        if label == "elements":
            env = RL.element_environment(env, RL.POSED_RHO)
        envs.append(posed(env, frames=frames))
    _model, _table, K = overlap_kernel(envs, fs)
    models["posed overlap, " + label] = choose_lam(K, rt, lams=RL.WIDE_LAMS, folds=10, seed=0,
                                                   name=label)

print()
print("%-26s %8s %12s %10s" % ("model", "q2", "error (ppm)", "MAE (ppm)"))
for name, r in models.items():
    print("%-26s %+8.3f %12.2f %10.2f" % (name, r.q2, r.error, r.mae))
print()
for a, b in (("posed overlap, fields", "HOSE lookup"), ("posed overlap, elements", "HOSE lookup"),
             ("posed overlap, fields", "posed overlap, elements")):
    d = paired_interval(models[a], models[b], rt, runs=2000, seed=0)
    print("%-24s minus %-24s %+.3f [%+.3f, %+.3f]" % (a, b, *d))
print("(%.0f s)" % (time.time() - start))

531 ring carbons in 73 molecules



model                            q2  error (ppm)  MAE (ppm)
HOSE lookup                  +0.696         8.55       5.88
posed overlap, fields        +0.906         4.75       3.35
posed overlap, elements      +0.861         5.78       4.31



posed overlap, fields    minus HOSE lookup              +0.210 [+0.147, +0.300]


posed overlap, elements  minus HOSE lookup              +0.165 [+0.103, +0.256]


posed overlap, fields    minus posed overlap, elements  +0.045 [+0.026, +0.067]
(83 s)


On these 531 ring carbons **both posed models beat the lookup by a wide margin**: +0.906 and
+0.861 against +0.696. Ring carbons are alike in type, so the lookup finds few close copies of a
given carbon's surroundings in 73 molecules. The posed overlap instead sees where each
substituent sits relative to the carbon: ortho, meta or para, near or far.

Here the fields are also ahead of the element channels, by +0.045 [+0.026, +0.067]. Hold that
number: the next two sections test whether it holds.

## The whole set, from the stored run

`benchmarks/run_local.py --only hose` measures the same comparisons on every carbon. It covers
the lookup at four depths, the two-point field model of plan item LP2, and the three-distance
kernel of item LP5, each with fields and with element channels. It also runs all 1,225 ring
carbons of LP3. It takes about an hour, so the cell reads what it stored. All differences are
in q2, with a 95 % interval bootstrapped over molecules.

In [4]:
with open("benchmarks/results/local/hose.json", encoding="utf-8") as fh:
    h = json.load(fh)

def show(rows):
    for r in rows:
        print("  %-48s %+7.3f %8.2f %8.2f" % (r["name"], r["q2"], r["error"], r["mae"]))

def diff(d):
    return "%+.3f [%+.3f, %+.3f]" % tuple(d)

s, a = h["shifts"], h["aromatic"]
print("every measured carbon: %d in %d molecules" % (s["n_samples"], s["n_molecules"]))
print("  %-48s %7s %8s %8s" % ("model", "q2", "error", "MAE"))
show([r for r in s["lookups"] if r["depth"] == h["main"]] + [s["counts"]]
     + s["two_point"] + s["three_distance"])
print()
print("against the lookup:")
for key, d in s["against_hose"].items():
    print("  %-38s %s" % (key.replace("_", " "), diff(d)))
print("three-distance, fields against elements:  " + diff(s["fields_against_elements"]))
print()
print("aromatic ring carbons: %d in %d molecules" % (a["n_samples"], a["n_molecules"]))
show([a["hose"], a["counts"]] + a["posed"])
print()
for key, d in a["against_hose"].items():
    print("  %-38s %s" % ((key + " against the lookup").replace("_", " "), diff(d)))
print("posed overlap, fields against elements:   " + diff(a["fields_against_elements"]))

every measured carbon: 5096 in 400 molecules
  model                                                 q2    error      MAE
  HOSE lookup, up to 4 bonds                        +0.976     8.34     4.93
  atom-centred counts                               +0.853    20.46    14.83
  two-point local field, rho 0.5                    +0.962    10.40     7.20
  two-point local field, rho 1                      +0.973     8.82     6.08
  three-distance, fields                            +0.990     5.45     3.55
  three-distance, elements                          +0.989     5.47     3.59
  three-distance, fields and elements               +0.992     4.83     3.24

against the lookup:
  counts                                 -0.123 [-0.136, -0.111]
  two point 0.5                          -0.014 [-0.018, -0.009]
  two point 1                            -0.003 [-0.008, +0.001]
  three distance fields                  +0.014 [+0.011, +0.017]
  three distance elements                +0.014 [+0.011, +

On every carbon the picture has three parts.

- **The two-point field model of LP2 only ties the lookup**: -0.003 [-0.008, +0.001] at its best
  envelope width, and it loses at the default width. LP2 reported a gain of +0.120 over the count
  reference, but that reference is itself 0.123 behind the lookup. The gain measured a weak
  reference.
- **The three-distance kernel of LP5 beats the lookup**, by +0.014 [+0.011, +0.017] in q2: 5.45
  ppm against 8.34, and a mean absolute error of 3.55 against 4.93 ppm. It sees angles, not only
  distances. But element channels in place of the fields do exactly as well (+0.000).
- **On all 1,225 ring carbons the posed overlap beats the lookup by +0.119 [+0.088, +0.153]**,
  and the element channels again do as well as the fields: -0.004 [-0.018, +0.011]. The +0.045
  of the live cell is not there.

So what beats the classical method is where the atoms around the carbon sit. What the fields say
about those atoms does no better than their elements.

## Is the lead of the live cell about a small training set?

The live cell and the stored run disagree about the fields against the element channels. On the
first 200 molecules of the file the fields were ahead, with an interval clear of zero; on the
whole set the two tie. One explanation is easy to believe: fields say in advance that atoms of
similar charge and size act alike, so they should help most when there are few examples to
learn from.

If that were the reason, any training set of the same size would show it, not only this one.
`run_local.py --only hose-curve` tests that on all ring carbons. Each fold keeps its test part
and trains on a random 25 %, 50 % or 100 % of its training molecules, five times over, and each
model's ridge is the one that cross-validates best at that size.

In [5]:
with open("benchmarks/results/local/hose_curve.json", encoding="utf-8") as fh:
    c = json.load(fh)
print("%7s %10s %9s %10s %8s   %s" % ("share", "molecules", "lookup", "elements", "fields",
                                     "fields minus elements (lowest to highest of 5 draws)"))
for r in c["rows"]:
    d = r["difference"]
    print("%7.2f %10d %+9.3f %+10.3f %+8.3f   %+.3f (%+.3f to %+.3f), interval clear for the "
          "fields in %d of %d draws" % (r["share"], r["train_molecules"], r["hose"], r["elements"],
                                        r["fields"], d[0], d[1], d[2], r["clear_for_fields"],
                                        c["draws"]))

  share  molecules    lookup   elements   fields   fields minus elements (lowest to highest of 5 draws)
   0.25         38    +0.644     +0.818   +0.829   +0.011 (-0.027 to +0.112), interval clear for the fields in 1 of 5 draws
   0.50         76    +0.716     +0.870   +0.874   +0.003 (-0.010 to +0.011), interval clear for the fields in 0 of 5 draws
   1.00        153    +0.780     +0.903   +0.899   -0.004 (-0.004 to -0.004), interval clear for the fields in 0 of 5 draws


**The explanation fails.** With 76 training molecules per fold, about the size of the live cell,
the fields and the element channels tie in every one of five random draws: +0.003 on average,
from -0.010 to +0.011, with no interval clear of zero. With 38 the mean is +0.011, but one draw
carries it: there the element model fell to +0.719 while the fields held. The other four draws
run from -0.027 to +0.001.

So the +0.045 of the live cell belongs to those particular 73 molecules, trained on each other.
It does not belong to the fields, and it does not belong to the size of the training set. A
margin read on one split can be a property of that split. Repeating the draw is what tells the
two apart, and it costs a few minutes.

What does grow as the data shrink is the lead of the geometric models over the lookup: +0.119
with 153 molecules, +0.185 with 38. A lookup needs a near copy of a carbon's surroundings among
the training carbons. A kernel can use a partial likeness.

## What to take from this notebook

- A target that belongs to one atom has a classical reference: the HOSE lookup, which needs no
  geometry and fits nothing. Measure it first. Here it reaches q2 +0.976, and the count
  reference of plan item LP2 is 0.123 behind it.
- The two-point field model only ties the lookup. Models that see where the atoms around the
  carbon sit beat it: the three-distance kernel by +0.014 on every carbon (5.45 against 8.34
  ppm), and the posed overlap by +0.119 on ring carbons, more when the training set is small.
- In both of those, five element channels in place of the five fields do as well. For a 13C
  shift, the positions of the neighbours matter. Their charge, size and lipophilicity add
  nothing that their elements do not already say.
- A margin read on one subset (+0.045 here) disappeared on random subsets of the same size.
  Repeat the draw before reading a margin.

From the command line, the posed overlap on the ring carbons:

```
cmf local examples/local/shifts_cdcl3.sdf --target shift --rho 0.25 --align
```

Its ring rule is broader than the one used here. It also takes the carbons where two rings meet,
1,405 carbons in 195 molecules, and gives q2 +0.877 on them. The numbers of this notebook use the
stricter rule of plan item LP3 (1,225 carbons). On that broader set `--reference` does not compare
with the counts, because the rule leaves out carbons the count model was fitted on.

and the three-distance kernel on every carbon:

```
cmf local examples/local/shifts_cdcl3.sdf --target shift --rho 0.5 --triangles --r-max 5 --reference
```

The HOSE lookup and the element channels are in `benchmarks/run_local.py`
(`environment_ids`, `hose_lookup`, `element_environment`); the command line offers the fields.

Where to read further in the manual: *An atom instead of a molecule: the local invariant
kernel* has the envelope and this comparison; *A co-field around one atom: the local alignment*
has the pose; *Three atoms at a time* has the three-distance kernel.